# An unknown reference code is refused

The producer sends a code outside the declared vocabulary.

## Prepare the model

Run this cell first. It imports the required types and defines the model used by this example. Start with a fresh repository Python kernel so declarations from other examples do not enter this model.

In [1]:
from pydantic import Field, ValidationError

from aoa.action_machine.domain import BaseCoreEntity, BaseEnumEntity, value
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.entity import entity


class ShopDomain(BaseDomain):
    """Group the shop declarations."""

    name = "shop"
    description = "Shop model"


@entity(description="Currencies accepted by the shop", domain=ShopDomain)
class CurrencyEnumEntity(BaseEnumEntity):
    """Declare the shop's currency vocabulary."""

    symbol: str = Field(description="Currency symbol")

    EUR = value(code="eur", display_name="Euro", symbol="€")
    USD = value(code="usd", display_name="US dollar", symbol="$")


@entity(description="Price tag", domain=ShopDomain)
class PriceTagCoreEntity(BaseCoreEntity):
    """Associate an amount with its currency."""

    id: str = Field(description="Tag identifier")
    amount: float = Field(description="Price amount")
    currency: CurrencyEnumEntity = Field(description="Money unit")

## Run the example

The following cell performs the operation described above. Its assertions check the result; any expected refusal is caught explicitly.

In [2]:
try:
    PriceTagCoreEntity(id="tag-1", amount=1.0, currency={"code": "unknown"})
except ValidationError as error:
    print("Field:", error.errors()[0]["loc"][0])
    print(error.errors()[0]["msg"])
else:
    raise AssertionError("The invalid reference was accepted")

Field: currency
Value error, 'unknown' is not a declared value of CurrencyEnumEntity. Declared: ['eur', 'usd'].


## What the result means

The reference has the right general shape, but its code is absent from the closed catalogue. Validation rejects the field instead of creating a value from the submitted description. Add a new member in the dictionary declaration if the domain must accept that code.